# PDF → graph → semantic search with Grafito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpmanson/GrafitoDB/blob/main/examples/semantic/pdf_chunking_colab.ipynb)

A minimal pipeline over a real Anthropic PDF —
**[Building Effective AI Agents](https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf)**:

PDF → Markdown → `Document / Section / Chunk` → embeddings → `search` → `expand` + `pack`.

Two design choices in this notebook:

- **No prior knowledge of the document.** We do not hardcode section titles. The PDF is
  turned into Markdown by [LiteParse](https://github.com/run-llama/liteparse) — a fast,
  fully local parser (no API key, no cloud) that recovers the real heading hierarchy. A
  tiny generic clean-up drops repeated running headers. `DocumentIngestor` consumes the
  Markdown, so you can swap LiteParse for any PDF→Markdown tool.
- **Readable graphs.** Instead of drawing the hundreds of chunks, we show the section tree
  and, per query, only the local **neighborhood** of the hit.

> In Grafito **1 node = 1 vector**: a long PDF is many linked passages, not a multi-vector
> single node.


In [ ]:
%pip install -q "grafitodb[viz]>=0.4.0" liteparse sentence-transformers networkx requests
print("OK")


## 1. Visualization helpers


In [ ]:
from __future__ import annotations

import re
import tempfile
from pathlib import Path

import liteparse
import requests
from IPython.display import HTML, Markdown, display

from grafito import GrafitoDatabase
from grafito.document import DocumentIngestor, MarkdownChunker, TitleContextEnricher
from grafito.embedding_functions.base import EmbeddingFunction
from grafito.integrations import save_pyvis_html

COLORS = {"Document": "#264653", "DocumentVersion": "#2a9d8f",
          "Section": "#e9c46a", "Chunk": "#f4a261", "hit": "#e63946"}


def _label(node_id, attrs):
    p, labels = attrs.get("properties") or {}, attrs.get("labels") or []
    if "Document" in labels:
        return f"PDF: {p.get('title') or node_id}"[:40]
    if "DocumentVersion" in labels:
        return f"v{p.get('generation', '?')}"
    if "Section" in labels:
        return f"§ {(p.get('title') or '?')[:28]}"
    if "Chunk" in labels:
        return f"#{p.get('global_seq', '?')} {(p.get('text') or '')[:26]}…".replace(chr(10), " ")
    return str(node_id)


def show_graph(db, ids, *, title="", hits=None, height="460px"):
    """Draw only `ids` (a small subgraph). `hits` are painted red."""
    hits = hits or set()
    G = db.to_networkx().subgraph(ids).copy()
    for nid in G.nodes:
        a = G.nodes[nid]
        labels = a.get("labels") or []
        a["properties"] = {**(a.get("properties") or {}),
                           "_c": COLORS["hit"] if nid in hits else COLORS.get(labels[0] if labels else "", "#8ecae6")}
    path = Path(tempfile.gettempdir()) / "g.html"
    save_pyvis_html(G, path=str(path), notebook=False, directed=True, color_by_label=False,
                    node_color_attr="_c", label_fn=_label, physics="spread", height=height,
                    width="100%", bgcolor="#fff", font_color="#222", cdn_resources="in_line")
    if title:
        display(Markdown(f"### {title}"))
    display(HTML(f'<div style="border:1px solid #ddd;border-radius:8px">{path.read_text()}</div>'))


def legend():
    html = "".join(f'<span style="margin-right:14px"><span style="display:inline-block;width:11px;'
                   f'height:11px;background:{c};border-radius:50%;margin-right:5px"></span>{n}</span>'
                   for n, c in COLORS.items())
    display(HTML(f"<div style='font:14px sans-serif'>{html}</div>"))


legend()


## 2. PDF → Markdown with LiteParse

[LiteParse](https://github.com/run-llama/liteparse) is a fast, local PDF parser that emits
Markdown with a real heading hierarchy — so we don't hardcode any section titles. We only
apply a small **generic** clean-up: drop repeated `Chapter N` running labels and a heading
that duplicates the previous one (a large divider *banner* plus the running header). OCR is
off because this PDF is vector text (parses in ~0.1 s).


In [ ]:
PDF_URL = ("https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-"
           "%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf")
PDF_PATH = Path("agents.pdf")
PDF_PATH.write_bytes(requests.get(PDF_URL, timeout=120).content)


def clean_markdown(md_text):
    """Drop generic running-header noise (repeated 'Chapter N' + duplicate headings)."""
    out, prev = [], None
    for line in md_text.splitlines():
        s = line.strip()
        if s.startswith("#"):
            body = s.lstrip("#").strip()
            if re.fullmatch(r"Chapter \d+", body, re.I):        # running chapter label
                continue
            low = body.lower().rstrip(":").strip()
            if prev and (low == prev or low.startswith(prev) or prev.startswith(low)):
                continue                                        # banner / running-header dup
            out.append(line)
            prev = low
        else:
            if s:
                prev = None
            out.append(line)
    return "\n".join(out)


parser = liteparse.LiteParse(output_format="markdown", ocr_enabled=False)
result = parser.parse(str(PDF_PATH))
pages = [result.get_page(i) for i in range(1, result.num_pages + 1)]
DOC_TEXT = clean_markdown("\n\n".join(p.markdown for p in pages if p))

heads = [ln for ln in DOC_TEXT.splitlines() if ln.startswith("#")]
print(f"{len(DOC_TEXT):,} chars · {len(heads)} headings\n")
print("\n".join(heads[:30]))


## 3. Ingest: hierarchical chunking + embeddings


In [ ]:
from sentence_transformers import SentenceTransformer


class STEmbedder(EmbeddingFunction):
    def __init__(self, model="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name, self.model = model, SentenceTransformer(model)
        self._dim = int(self.model.get_sentence_embedding_dimension())

    def __call__(self, input):
        return self.model.encode(input, normalize_embeddings=True).tolist()

    @staticmethod
    def name():
        return "st_notebook"

    def default_space(self):
        return "cosine"

    def supported_spaces(self):
        return ["cosine"]

    @staticmethod
    def build_from_config(c):
        return STEmbedder(c.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"))

    def get_config(self):
        return {"model_name": self.model_name, "dim": self._dim}

    @staticmethod
    def validate_config(c):
        return None

    @property
    def dimension(self):
        return self._dim


db = GrafitoDatabase(":memory:")
db.create_vector_index("chunks", backend="bruteforce", embedding_function=STEmbedder(),
                       options={"metric": "cosine"})

ing = DocumentIngestor(db, chunker=MarkdownChunker(max_chars=1100, overlap=120),
                       embed_index="chunks", configure_fts=db.has_fts5(),
                       enricher=TitleContextEnricher(), hierarchy="auto")

DOC_KEY = "anthropic/building-effective-ai-agents"
result = ing.ingest(DOC_TEXT, document_key=DOC_KEY, title="Building Effective AI Agents",
                    source=PDF_URL, embed=True)
parent = db.match_nodes(labels=["Document"], properties={"document_key": DOC_KEY}, limit=1)[0]
print(f"sections={result.n_sections}  passages={result.n_passages}  hierarchy={result.hierarchy}")


## 4. Structure: the section tree (without the chunks)


In [ ]:
struct_ids = {parent.id} | {
    n.id for n in db.match_nodes(properties={"managed_by": "grafito.document"})
    if n.properties.get("owner_document_id") == parent.id
    and ("Section" in n.labels or "DocumentVersion" in n.labels)
}
show_graph(db, struct_ids, title="Document → Version → Sections", height="560px")

print("ToC:")
for sec in ing.toc(DOC_KEY):
    print(f"  {sec.title}")
    for c in sec.children[:8]:
        print(f"     · {c.title}")


## 5. Semantic search + neighborhood

Each query draws **only** the hit, its section/ancestors and neighboring passages —
not the whole graph.


In [ ]:
QUERIES = [
    "How do AI agents differ from traditional automation?",
    "When to use a single agent vs multi-agent orchestration?",
    "What are agent Skills and when to use them?",
    "customer support use cases for AI agents",
]


def neighborhood(hits, window=1):
    ids = {parent.id}
    for h in hits:
        ids.add(h.node.id)
        ex = ing.expand(h.node, window=window, include_ancestors=True)
        ids.update(p.id for p in ex.passages)
        ids.update(a.id for a in ex.ancestors)
        if ex.section:
            ids.add(ex.section.id)
    return ids


def run_query(query, k=3, draw=True):
    hits = ing.search(query, k=k)
    print(f"QUERY: {query}")
    for i, h in enumerate(hits, 1):
        print(f"  {i}. score={h.score:.3f} seq={h.global_seq}  "
              f"{(h.node.properties.get('text') or '')[:150].strip()}…")
    if draw and hits:
        show_graph(db, neighborhood(hits), title=query, hits={h.node.id for h in hits})
    return hits


run_query(QUERIES[0])


Try another (change the index):


In [ ]:
run_query(QUERIES[1], k=4)


## 6. Expand + pack (context for an LLM)


In [ ]:
hits = ing.search(QUERIES[1], k=3)
expanded = ing.expand(hits[0].node, window=1, include_parent=True, include_ancestors=True)
packed = ing.pack(expanded, max_chars=2000, include_citations=True)

print("section:", expanded.section and expanded.section.properties.get("title"))
print("ancestors:", [a.properties.get("title") for a in expanded.ancestors])
print("window seqs:", [p.properties.get("global_seq") for p in expanded.passages])
print("\n--- PACKED ---\n")
print(packed.text[:1800])

show_graph(db, neighborhood(hits[:1], window=1), title="Expand window",
           hits={hits[0].node.id})


## 7. Hybrid search (vector + FTS + RRF)


In [ ]:
if db.has_fts5():
    for h in ing.hybrid_search(QUERIES[2], k=3):
        print(f"  rrf={h.score:.4f}  {(h.node.properties.get('text') or '')[:150].strip()}…")
else:
    print("FTS5 unavailable — skipping hybrid search.")


## 8. Exercises

1. Run the pipeline on another PDF: nothing to change — LiteParse recovers its headings.
2. `hierarchy=False` in a new `DocumentIngestor` → a flat-chunk graph.
3. `window=0` vs `window=2` in `expand` on the same query.
4. (Advanced) `tree_select` with an LLM that picks `node_key`s from the ToC.

**Refs:** [Document Chunking](https://jpmanson.github.io/GrafitoDB/search/document-chunking/) ·
[LiteParse](https://github.com/run-llama/liteparse) ·
[Visualization](https://jpmanson.github.io/GrafitoDB/integrations/visualization/)


In [ ]:
db.close()
